# Asignador de portafolio

Jose Luis Sanchez Davila · jlsd

Herramienta para decidir como repartir una inversion entre activos que se pueden comprar desde Mexico, todo medido en pesos. Descarga precios, estima el riezgo de cada activo con GARCH y DCC, arma la cartera, la compara contra alternativas simples en un backtest fuera de muestra y al final da las ordenes de compra y venta mas un reporte en HTML.

Se cambian los parametros de la siguiente celda y se corre todo, o doble clic en actualizar.bat. Combiene correrlo despues de las 3 pm cuando ya cerro el mercado. No es asesoria financiera.

Parametros: CAPITALMXN es el valor total de la cuenta. POSICIONESMXN es lo que ya tienes invertido en pesos con las claves IVV, QQQ, NAFTRAC, GLD, IEF, BIL y CETES (vacio si todo esta en efectivo). ESTRATEGIA en "auto" escoge la de mejor Sharpe. VOLOBJETIVO es la volatilidad anual que aguantas, PESOMAX el tope por activo, BANDA el movimiento minimo para operar, COMISION e IVA son los de GBM (0.25% mas IVA) y UMBRALREBALANCEO la desviacion para reajustar. El token de Banxico se lee solo de tokenbanxico.txt.

In [1]:
# jlsd
# parametros del inversionista
CAPITALMXN = 100000
POSICIONESMXN = {}
ESTRATEGIA = "auto"
VOLOBJETIVO = 0.10
PESOMAX = 0.40
COMISION, IVA = 0.0025, 0.16
UMBRALREBALANCEO = 0.05
BANDA = 0.05
BANXICOTOKEN = ""
TEMA = "oscuro"
INICIO, INICIOOOS = "2007-01-01", "2012-01-01"

In [2]:
# librerias
import os, re, json, time, logging, platform, shutil, subprocess, warnings
from datetime import datetime, date
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats, optimize
from scipy.special import xlogy
import arch
import plotly
import plotly.offline
import plotly.graph_objects as go
import plotly.io as pio
import great_tables
from great_tables import GT, loc, style
from IPython.display import display, HTML, Markdown

warnings.filterwarnings("ignore")
logging.getLogger("yfinance").setLevel(logging.CRITICAL)
archivotoken = Path.cwd() / "tokenbanxico.txt"
BANXICOTOKEN = BANXICOTOKEN or os.environ.get("BANXICOTOKEN", "") or (archivotoken.read_text(encoding="utf8").strip() if archivotoken.exists() else "")

# carpetas del proyecto
RAIZ = Path.cwd()
DIRCACHE, DIRPROC = RAIZ / "datos" / "cache", RAIZ / "datos" / "procesados"
DIROUT = RAIZ / "resultados"
DIRHIST = DIROUT / "decisiones"
for d in (DIRCACHE, DIRPROC, DIROUT, DIRHIST):
    d.mkdir(parents=True, exist_ok=True)

# activos que se pueden comprar desde mexico
ACTIVOS = {
    "IVV": dict(nombre="S&P 500", ticker="IVV", moneda="USD"),
    "QQQ": dict(nombre="Nasdaq 100", ticker="QQQ", moneda="USD"),
    "NAFTRAC": dict(nombre="IPC Mexico", ticker="NAFTRACISHRS.MX", moneda="MXN"),
    "GLD": dict(nombre="Oro", ticker="GLD", moneda="USD"),
    "IEF": dict(nombre="Bonos del Tesoro 7 a 10 años", ticker="IEF", moneda="USD"),
    "BIL": dict(nombre="Dolares en T Bills", ticker="BIL", moneda="USD"),
    "CETES": dict(nombre="CETES 28 dias", ticker="Banxico / FRED", moneda="MXN"),
}
ALTERNOS = {"NAFTRAC": "^MXX"}
RIESGOSOS = ["IVV", "QQQ", "NAFTRAC", "GLD", "IEF", "BIL"]
ACT = RIESGOSOS + ["CETES"]
N = len(RIESGOSOS)
COSTO = COMISION * (1 + IVA)
COSTOVEC = np.r_[np.full(N, COSTO), 0.0]
ESTRATEGIAS = ["60/40 con freno de riesgo", "Control de volatilidad (bandas)", "Control de volatilidad", "Paridad de riesgo (DCC)",
               "Paridad de riesgo (historica)", "Minima varianza (DCC)", "1/N", "60/40 Mexico", "Solo IPC", "Solo CETES"]
CONBANDAS = {"Control de volatilidad (bandas)": "Control de volatilidad"}
assert ESTRATEGIA == "auto" or ESTRATEGIA in ESTRATEGIAS, f"ESTRATEGIA debe ser 'auto' o una de: {ESTRATEGIAS}"

# colores y estilo de las graficas
TEMAS = {
    "claro": dict(sup="#fcfcfb", plano="#f9f9f7", txt="#0b0b0b", txt2="#52514e", mudo="#898781", rejilla="#e1e0d9", eje="#c3c2b7",
                  borde="rgba(11,11,11,0.10)", banda="rgba(42,120,214,0.14)", banda2="rgba(42,120,214,0.26)",
                  serie=["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"],
                  div=["#2a78d6", "#f0efec", "#e34948"], seq=["#cde2fb", "#86b6ef", "#2a78d6", "#184f95"]),
    "oscuro": dict(sup="#1a1a19", plano="#0d0d0d", txt="#f0efec", txt2="#c3c2b7", mudo="#898781", rejilla="#2c2c2a", eje="#383835",
                   borde="rgba(255,255,255,0.10)", banda="rgba(57,135,229,0.16)", banda2="rgba(57,135,229,0.30)",
                   serie=["#3987e5", "#d95926", "#199e70", "#c98500", "#d55181", "#0ca30c", "#9085e9", "#e66767"],
                   div=["#3987e5", "#383835", "#e66767"], seq=["#184f95", "#256abf", "#3987e5", "#86b6ef"]),
}
T = TEMAS[TEMA]
ESTADO = {"verde": "#0ca30c", "amarillo": "#fab219", "naranja": "#ec835a", "rojo": "#d03b3b"}
COLORACT = dict(zip(ACT, T["serie"]))
COLOREST = dict(zip(["60/40 con freno de riesgo", "Control de volatilidad (bandas)", "Control de volatilidad", "60/40 Mexico", "1/N", "Solo IPC",
                     "Paridad de riesgo (DCC)", "Paridad de riesgo (historica)"], T["serie"])) | {"Minima varianza (DCC)": T["mudo"], "Solo CETES": T["mudo"]}
LINEAEST = {"Minima varianza (DCC)": "dash", "Solo CETES": "dot"}
FUENTE = "Times New Roman, Times, serif"
FUENTESGT = ["Times New Roman", "Times", "serif"]

pio.templates["asignador"] = go.layout.Template(layout=dict(
    paper_bgcolor=T["sup"], plot_bgcolor=T["sup"], colorway=T["serie"],
    font=dict(family=FUENTE, color=T["txt2"], size=13),
    title=dict(font=dict(color=T["txt"], size=18), x=0.01, xanchor="left", y=0.96, yanchor="top"),
    xaxis=dict(showgrid=False, linecolor=T["eje"], ticks="outside", tickcolor=T["eje"], zeroline=False),
    yaxis=dict(gridcolor=T["rejilla"], showline=False, zerolinecolor=T["eje"], ticks=""),
    legend=dict(orientation="h", yanchor="bottom", y=1.01, x=0, bgcolor="rgba(0,0,0,0)", font=dict(color=T["txt2"])),
    hoverlabel=dict(bgcolor=T["plano"], bordercolor=T["eje"], font=dict(family=FUENTE, color=T["txt"])),
    margin=dict(l=70, r=40, t=110, b=60), height=470,
))
pio.templates.default = "asignador"
FIGURAS, TABLAS, TARJETAS = [], [], []

def titulo(fig, t, sub=""):
    fig.update_layout(title=dict(text=f"<b>{t}</b>" + (f"<br><span style='font-size:13px;color:{T['txt2']}'>{sub}</span>" if sub else "")))
    return fig

def mostrar(fig, nombre):
    fig.add_annotation(text="jlsd", xref="paper", yref="paper", x=1, y=1, xanchor="right", yanchor="bottom", yshift=8, showarrow=False,
                       font=dict(size=10, color=T["mudo"]), opacity=0.8)
    FIGURAS.append((nombre, fig))
    fig.show()

def tabla(df, t, sub=""):
    g = (GT(df).tab_header(title=t, subtitle=sub)
         .tab_options(table_background_color=T["sup"], table_font_color=T["txt"], table_font_names=FUENTESGT, table_font_size="14px",
                      heading_background_color=T["sup"], heading_align="left", heading_title_font_size="18px", heading_subtitle_font_size="13px",
                      column_labels_background_color=T["sup"], column_labels_font_weight="600", column_labels_font_size="13px",
                      table_border_top_color=T["eje"], table_border_bottom_color=T["eje"], heading_border_bottom_color=T["eje"],
                      column_labels_border_top_color=T["eje"], column_labels_border_bottom_color=T["eje"], table_body_hlines_color=T["rejilla"],
                      table_body_border_bottom_color=T["eje"], source_notes_background_color=T["sup"], source_notes_font_size="12px",
                      data_row_padding="7px", table_width="100%"))
    return g

def mostrartabla(g, nombre):
    g = g.tab_source_note("jlsd")
    TABLAS.append((nombre, g))
    display(g)

def tarjetas(items):
    partes = []
    for it in items:
        t, v = it[:2]
        fondo = it[2] if len(it) > 2 else None
        c1, c2 = ("#0b0b0b", "#0b0b0b") if fondo else (T["txt2"], T["txt"])
        partes.append(f"<div style='background:{fondo or T['sup']};border:1px solid {T['borde']};border-radius:12px;padding:14px 16px;font-family:{FUENTE}'>"
                      f"<div style='color:{c1};font-size:14px'>{t}</div>"
                      f"<div style='color:{c2};font-size:26px;font-weight:700;margin-top:6px;line-height:1.15'>{v}</div></div>")
    h = (f"<div style='display:grid;grid-template-columns:repeat(auto-fill,minmax(165px,1fr));gap:10px;background:{T['plano']};padding:10px;border-radius:14px'>"
         f"{''.join(partes)}</div><div style='color:{T['mudo']};font-size:10px;text-align:right;padding:2px 8px'>jlsd</div>")
    TARJETAS.append(h)
    display(HTML(h))

def mxn(x):
    return f"${x:,.0f}"

def tickslog(vmin, vmax, base):
    t = [float(base)]
    while t[-1] < vmax:
        t.append(t[-1] * 2)
    while t[0] > vmin:
        t.insert(0, t[0] / 2)
    return t

def etiquetasderecha(fig, puntos, ticks):
    ys = np.log10([v for nada, v, nada in puntos])
    sep = 0.055 * np.log10(ticks[-1] / ticks[0])
    prev, pos = None, {}
    for i in np.argsort(ys):
        pos[i] = ys[i] if prev is None else max(ys[i], prev + sep)
        prev = pos[i]
    for i, (texto, nada, color) in enumerate(puntos):
        fig.add_annotation(x=1, xref="paper", y=pos[i], yref="y", text=texto, showarrow=False, xanchor="left", xshift=10, font=dict(color=color, size=12))

print(f"Python {platform.python_version()} · pandas {pd.__version__} · arch {arch.__version__} · plotly {plotly.__version__} · great_tables {great_tables.__version__}")

Python 3.12.10 · pandas 2.3.3 · arch 9.0.0 · plotly 6.5.0 · great_tables 1.2.0


## 1. Datos en pesos

Precios de Yahoo Finance ajustados por dividendos. Lo que esta en dolares se pasa a pesos con el tipo de cambio de cada dia y CETES 28 dias sale de Banxico. Si alguna fuente falla se usa la ultima descarga guardada.

In [3]:
# descarga de precios con reintentos
def bajaryahoo(tickers, inicio, intentos=3):
    import yfinance as yf
    datos, pend = {}, dict(tickers)
    for i in range(intentos):
        try:
            raw = yf.download(list(pend.values()), start=inicio, auto_adjust=True, progress=False, threads=False)["Close"]
        except Exception:
            raw = pd.DataFrame()
        if isinstance(raw, pd.Series):
            raw = raw.to_frame(next(iter(pend.values())))
        for k, tk in list(pend.items()):
            if tk in raw.columns and raw[tk].notna().sum() > 250:
                datos[k] = raw[tk]
                pend.pop(k)
        if not pend:
            break
        time.sleep(3 * (i + 1))
    return datos, pend

# tasa de cetes, primero banxico y si no fred
def bajarcetes(inicio):
    if BANXICOTOKEN:
        import requests
        respaldo = None
        for sid, desc in [("SF43936", "CETES 28 dias, subasta semanal"), ("SF282", "CETES 28 dias, promedio mensual")]:
            try:
                url = f"https://www.banxico.org.mx/SieAPIRest/service/v1/series/{sid}/datos/{inicio}/{date.today():%Y-%m-%d}"
                j = requests.get(url, headers={"Bmx-Token": BANXICOTOKEN}, timeout=30).json()
                d = pd.DataFrame(j["bmx"]["series"][0]["datos"])
                s = pd.Series(pd.to_numeric(d["dato"].str.replace(",", ""), errors="coerce").values,
                              index=pd.to_datetime(d["fecha"], format="%d/%m/%Y")).dropna().sort_index()
                if len(s) < 100:
                    continue
                if (pd.Timestamp(date.today()) - s.index.max()).days <= 20:
                    return s, f"Banxico SIE {sid} · {desc}"
                respaldo = respaldo or (s, f"Banxico SIE {sid} · {desc}")
            except Exception as e:
                print(f"Banxico {sid} no respondio ({type(e).__name__})")
        if respaldo:
            return respaldo
    try:
        d = pd.read_csv("https://fred.stlouisfed.org/graph/fredgraph.csv?id=INTGSTMXM193N")
        s = pd.Series(pd.to_numeric(d.iloc[:, 1], errors="coerce").values, index=pd.to_datetime(d.iloc[:, 0])).dropna()
        s = s[s.index >= pd.Timestamp(inicio) - pd.DateOffset(months=2)]
        if len(s) > 24:
            return s.sort_index(), "FRED INTGSTMXM193N · CETES promedio mensual (FMI)"
    except Exception as e:
        print(f"FRED no respondio ({type(e).__name__})")
    return None, None

ahora = datetime.now()
metapath = DIRCACHE / "meta.json"
meta = json.loads(metapath.read_text(encoding="utf8")) if metapath.exists() else {}

# si yahoo falla se usa la cache
tickers = {k: ACTIVOS[k]["ticker"] for k in RIESGOSOS} | {"USDMXN": "MXN=X"}
datos, pend = bajaryahoo(tickers, INICIO)
usados = {k: tickers[k] for k in datos}
for k, alt in ALTERNOS.items():
    if k in pend:
        extra, nada = bajaryahoo({k: alt}, INICIO)
        if k in extra:
            datos[k], usados[k] = extra[k], alt
            pend.pop(k)
            print(f"{tickers[k]} no respondio; se usa {alt} (indice sin dividendos)")
if not pend:
    precios = pd.DataFrame(datos)
    precios.index = pd.to_datetime(precios.index).tz_localize(None)
    precios.index.name = "Fecha"
    precios.to_csv(DIRCACHE / "precios.csv")
    meta["precios"] = {"descarga": f"{ahora:%d/%m/%Y %H:%M}", "tickers": usados}
    ORIGENP = f"Yahoo Finance en vivo ({ahora:%d/%m/%Y %H:%M})"
elif (DIRCACHE / "precios.csv").exists():
    precios = pd.read_csv(DIRCACHE / "precios.csv", parse_dates=["Fecha"], index_col="Fecha")
    usados = meta["precios"]["tickers"]
    ORIGENP = f"cache del {meta['precios']['descarga']} (no respondieron: {', '.join(pend)})"
else:
    raise RuntimeError(f"Yahoo Finance no entrego {list(pend)} y no hay cache de precios")

cetes, fuentec = bajarcetes(INICIO)
if cetes is not None:
    cetes.rename("CETES").to_frame().to_csv(DIRCACHE / "cetes.csv", index_label="Fecha")
    meta["cetes"] = {"descarga": f"{ahora:%d/%m/%Y %H:%M}", "fuente": fuentec}
    ORIGENC = fuentec
elif (DIRCACHE / "cetes.csv").exists():
    cetes = pd.read_csv(DIRCACHE / "cetes.csv", parse_dates=["Fecha"], index_col="Fecha")["CETES"]
    ORIGENC = f"cache del {meta['cetes']['descarga']} · {meta['cetes']['fuente']}"
else:
    raise RuntimeError("No se pudo obtener la tasa de CETES y no hay cache")
metapath.write_text(json.dumps(meta, indent=1, ensure_ascii=False), encoding="utf8")

# se quita la sesion de hoy si no ha cerrado
fin = min(precios[c].last_valid_index() for c in precios.columns)
if fin.date() == ahora.date() and ahora.hour < 16:
    fin = precios.index[precios.index < pd.Timestamp(ahora.date())].max()
    print("El mercado no ha cerrado: se excluye la sesion de hoy")
precios = precios.loc[:fin]
dias = precios["IVV"].dropna().index
P = precios.reindex(dias).ffill(limit=5).dropna()

# todo a pesos
rlocal = P.pct_change()
R = pd.DataFrame({k: (1 + rlocal[k]) * (1 + rlocal["USDMXN"]) - 1 if ACTIVOS[k]["moneda"] == "USD" else rlocal[k] for k in RIESGOSOS})
tasa = cetes.reindex(P.index.union(cetes.index)).sort_index().ffill().reindex(P.index)
R["CETES"] = tasa.shift(1) / 100 * P.index.to_series().diff().dt.days / 360
R = R.dropna()
FECHAS = R.index
print(f"Precios: {ORIGENP}\nCETES: {ORIGENC}\nPanel en pesos: {len(R):,} dias, del {FECHAS[0]:%d/%m/%Y} al {FECHAS[-1]:%d/%m/%Y}")

Precios: Yahoo Finance en vivo (21/11/2026 18:09)
CETES: Banxico SIE SF43936 · CETES 28 dias, subasta semanal
Panel en pesos: 4,690 dias, del 31/05/2007 al 20/11/2026


In [4]:
# revision rapida de los datos
def saltos(r):
    r = r.dropna()
    return int((np.abs(r - r.mean()) > 8 * r.std()).sum())

calidad = pd.DataFrame({
    "Clave": RIESGOSOS + ["USDMXN", "CETES"],
    "Activo": [ACTIVOS[k]["nombre"] for k in RIESGOSOS] + ["Tipo de cambio", "CETES 28 dias"],
    "Fuente": [usados.get(k, ACTIVOS[k]["ticker"]) for k in RIESGOSOS] + [usados.get("USDMXN", "MXN=X"), ORIGENC.split(" · ")[0]],
    "Primer dato": [precios[k].first_valid_index().strftime("%d/%m/%Y") for k in RIESGOSOS + ["USDMXN"]] + [cetes.index.min().strftime("%d/%m/%Y")],
    "Ultimo dato": [precios[k].last_valid_index().strftime("%d/%m/%Y") for k in RIESGOSOS + ["USDMXN"]] + [cetes.index.max().strftime("%d/%m/%Y")],
    "Dias rellenados": [int(precios[k].reindex(dias).isna().sum()) for k in RIESGOSOS + ["USDMXN"]] + [0],
    "Saltos > 8σ": [saltos(rlocal[k]) for k in RIESGOSOS + ["USDMXN"]] + [0],
})
rezago = max((FECHAS[-1] - cetes.index.max()).days, 0)
g = (tabla(calidad, "Calidad de los datos", f"Panel del {FECHAS[0]:%d/%m/%Y} al {FECHAS[-1]:%d/%m/%Y} · {len(R):,} dias habiles de EUA")
     .tab_source_note(f"Los saltos mayores a 8σ son cambios diarios extremos en la moneda original: se señalan para revision, no se modifican. "
                      f"El ultimo dato de CETES tiene {rezago} dias de antiguedad; se mantiene vigente hasta que llegue uno nuevo."))
if rezago > 45:
    g = g.tab_style(style=[style.fill(color=ESTADO["naranja"]), style.text(color="#0b0b0b", weight="bold")], locations=loc.body(columns="Ultimo dato", rows=[len(calidad) - 1]))
mostrartabla(g, "calidad")

GT(_tbl_data=     Clave                        Activo               Fuente Primer dato  \
0      IVV                       S&P 500                  IVV  03/01/2007   
1      QQQ                    Nasdaq 100                  QQQ  03/01/2007   
2  NAFTRAC                    IPC Mexico      NAFTRACISHRS.MX  02/01/2007   
3      GLD                           Oro                  GLD  03/01/2007   
4      IEF  Bonos del Tesoro 7 a 10 años                  IEF  03/01/2007   
5      BIL            Dolares en T Bills                  BIL  30/05/2007   
6   USDMXN                Tipo de cambio                MXN=X  01/01/2007   
7    CETES                 CETES 28 dias  Banxico SIE SF43936  04/01/2007   

  Ultimo dato  Dias rellenados  Saltos > 8σ  
0  20/11/2026                0            4  
1  20/11/2026                0            4  
2  20/11/2026              372            3  
3  20/11/2026                0            2  
4  20/11/2026                0            0  
5  20/11/2026              101           12  
6  20/11/2026                7            4  
7  19/11/2026                0            0  , _body=<great_tables._gt_data.Body object at 0x000001B4AFE52720>, _boxhead=Boxhead([ColInfo(var='Clave', type=<ColInfoTypeEnum.default: 1>, column_label='Clave', column_align='left', column_width=None, stub_level=None), ColInfo(var='Activo', type=<ColInfoTypeEnum.default: 1>, column_label='Activo', column_align='left', column_width=None, stub_level=None), ColInfo(var='Fuente', type=<ColInfoTypeEnum.default: 1>, column_label='Fuente', column_align='left', column_width=None, stub_level=None), ColInfo(var='Primer dato', type=<ColInfoTypeEnum.default: 1>, column_label='Primer dato', column_align='right', column_width=None, stub_level=None), ColInfo(var='Ultimo dato', type=<ColInfoTypeEnum.default: 1>, column_label='Ultimo dato', column_align='right', column_width=None, stub_level=None), ColInfo(var='Dias rellenados', type=<ColInfoTypeEnum.default: 1>, column_label='Dias rellenados', column_align='right', column_width=None, stub_level=None), ColInfo(var='Saltos > 8σ', type=<ColInfoTypeEnum.default: 1>, column_label='Saltos > 8σ', column_align='right', column_width=None, stub_level=None)]), _stub=<great_tables._gt_data.Stub object at 0x000001B4AFFDF830>, _spanners=Spanners([]), _heading=Heading(title='Calidad de los datos', subtitle='Panel del 31/05/2007 al 20/11/2026 · 4,690 dias habiles de EUA', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x000001B4B038F140>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x000001B4B038ECF0>, _source_notes=['Los saltos mayores a 8σ son cambios diarios extremos en la moneda original: se señalan para revision, no se modifican. El ultimo dato de CETES tiene 1 dias de antiguedad; se mantiene vigente hasta que llegue uno nuevo.', 'jlsd'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x000001B4B038F6E0>, _formats=[], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='100%'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#1a1a19'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['Times New Roman', 'Times', 'serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='14px'), table_font_weight=OptionsInfo(scss=True, category='table', typ

## 2. Mercados en pesos

Como van los activos para alguien que mide su dinero en pesos.

In [5]:
# rendimientos por ventana
V = (1 + R).cumprod()
def ventana(n):
    return V.iloc[-1] / V.iloc[-1 - n] - 1 if len(V) > n else pd.Series(np.nan, index=V.columns)
inicioanio = V.loc[V.index < pd.Timestamp(FECHAS[-1].year, 1, 1)].iloc[-1]
mercado = pd.DataFrame({
    "Activo": [ACTIVOS[k]["nombre"] for k in ACT],
    "1 mes": ventana(21).values, "3 meses": ventana(63).values, "En el año": (V.iloc[-1] / inicioanio - 1).values,
    "12 meses": ventana(252).values, "3 años (anual)": ((1 + ventana(756)) ** (1 / 3) - 1).values,
    "Volatilidad 12m": (R.iloc[-252:].std() * np.sqrt(252)).values,
    "Caida desde maximo": (V.iloc[-1] / V.cummax().iloc[-1] - 1).values,
})
colspct = list(mercado.columns[1:])
g = (tabla(mercado, "Rendimientos", f"Al cierre del {FECHAS[-1]:%d/%m/%Y} · incluye dividendos y tipo de cambio")
     .fmt_percent(columns=colspct, decimals=1)
     .data_color(columns=["1 mes", "3 meses", "En el año", "12 meses", "3 años (anual)"], palette=T["div"], domain=[-0.3, 0.3], na_color=T["sup"], truncate=True)
     .cols_align(align="right", columns=colspct))
mostrartabla(g, "mercado")

base = CAPITALMXN * V / V.iloc[0]
fig = go.Figure()
for k in ACT:
    fig.add_scatter(x=base.index, y=base[k], name=ACTIVOS[k]["nombre"], line=dict(color=COLORACT[k], width=2),
                    hovertemplate="%{y:$,.0f}<extra>" + ACTIVOS[k]["nombre"] + "</extra>")
tv = tickslog(base.min().min(), base.max().max(), CAPITALMXN)
etiquetasderecha(fig, [(ACTIVOS[k]["nombre"], base[k].iloc[-1], T["txt2"]) for k in ACT], tv)
fig.update_yaxes(type="log", tickvals=tv, ticktext=[mxn(v) for v in tv])
fig.update_xaxes(range=[base.index[0], base.index[-1]])
fig.update_layout(hovermode="x unified", margin=dict(r=200))
titulo(fig, f"Que habria pasado con {mxn(CAPITALMXN)} en cada activo", f"Escala logaritmica, desde el {FECHAS[0]:%d/%m/%Y}")
mostrar(fig, "crecimientoactivos")

GT(_tbl_data=                         Activo     1 mes   3 meses  En el año  12 meses  \
0                       S&P 500 -0.076127  0.023525   0.084291  0.097094   
1                    Nasdaq 100 -0.117278 -0.026778   0.055265  0.060725   
2                    IPC Mexico -0.012253  0.052716   0.095891  0.143758   
3                           Oro -0.105742 -0.146013  -0.134952 -0.103723   
4  Bonos del Tesoro 7 a 10 años  0.030561  0.123742   0.046245  0.026868   
5            Dolares en T Bills  0.025357  0.138884   0.096261  0.079426   
6                 CETES 28 dias  0.005074  0.015337   0.060103  0.069029   

   3 años (anual)  Volatilidad 12m  Caida desde maximo  
0        0.206881         0.169648           -0.086350  
1        0.209070         0.241145           -0.154694  
2        0.130943         0.164290           -0.044404  
3        0.250992         0.291067           -0.275263  
4        0.071559         0.115611           -0.323811  
5        0.078402         0.095309           -0.094440  
6        0.092068         0.002570            0.000000  , _body=<great_tables._gt_data.Body object at 0x000001B4B02CE990>, _boxhead=Boxhead([ColInfo(var='Activo', type=<ColInfoTypeEnum.default: 1>, column_label='Activo', column_align='left', column_width=None, stub_level=None), ColInfo(var='1 mes', type=<ColInfoTypeEnum.default: 1>, column_label='1 mes', column_align='right', column_width=None, stub_level=None), ColInfo(var='3 meses', type=<ColInfoTypeEnum.default: 1>, column_label='3 meses', column_align='right', column_width=None, stub_level=None), ColInfo(var='En el año', type=<ColInfoTypeEnum.default: 1>, column_label='En el año', column_align='right', column_width=None, stub_level=None), ColInfo(var='12 meses', type=<ColInfoTypeEnum.default: 1>, column_label='12 meses', column_align='right', column_width=None, stub_level=None), ColInfo(var='3 años (anual)', type=<ColInfoTypeEnum.default: 1>, column_label='3 años (anual)', column_align='right', column_width=None, stub_level=None), ColInfo(var='Volatilidad 12m', type=<ColInfoTypeEnum.default: 1>, column_label='Volatilidad 12m', column_align='right', column_width=None, stub_level=None), ColInfo(var='Caida desde maximo', type=<ColInfoTypeEnum.default: 1>, column_label='Caida desde maximo', column_align='right', column_width=None, stub_level=None)]), _stub=<great_tables._gt_data.Stub object at 0x000001B4AFE52510>, _spanners=Spanners([]), _heading=Heading(title='Rendimientos', subtitle='Al cierre del 20/11/2026 · incluye dividendos y tipo de cambio', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x000001B4B038D760>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x000001B4B038E9F0>, _source_notes=['jlsd'], _footnotes=[], _styles=[StyleInfo(locname=LocBody(columns='1 mes', rows=[0], mask=None), grpname=None, colname='1 mes', rownum=0, colnum=None, styles=[CellStyleText(color='#FFFFFF', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None), CellStyleFill(color='#424b5e')]), StyleInfo(locname=LocBody(columns='1 mes', rows=[1], mask=None), grpname=None, colname='1 mes', rownum=1, colnum=None, styles=[CellStyleText(color='#FFFFFF', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None), CellStyleFill(color='#455575')]), StyleInfo(locname=LocBody(columns='1 mes', rows=[2], mask=None), grpname=None, colname='1 mes', rownum=2, colnum=None, styles=[CellStyleText(color='#FFFFFF', font=None, size=None, align=None, v_align=None, style=None, weight=None, stretch=None, decorate=None, transform=None, whitespace=None), CellStyleFill(color='#3a3b3b')]), StyleInfo(locname=LocBody(columns='1 mes', rows=[3], mask=None), grpname=None, colname='1 mes', rownum=3, colnum=None, styles=[CellStyleText(color='#FFFFFF', font=None, size=None, align=None, v_align=None, style=None, weigh

## 3. Motor de riesgo

GJR GARCH con distribucion t para cada activo y DCC para las correlaciones. Se reestima cada año usando solo datos pasados, entonces todo el backtest es fuera de muestra. De aqui sale la covarianza a 21 dias para armar la cartera y la de un dia para el VaR.

In [6]:
X = 100 * R[RIESGOSOS]
TT = len(X)
k0 = FECHAS.searchsorted(pd.Timestamp(INICIOOOS))
CORTES = list(range(k0, TT, 252))
H = 21
S1, S21, MU, NU = (np.full((TT, N), np.nan) for nada in range(4))
RC = np.full((TT, N, N), np.nan)

# dcc de engle
def recursionq(Z, a, b, Qbar):
    Q = np.empty((len(Z) + 1, N, N))
    Q[0] = Qbar
    c = (1 - a - b) * Qbar
    for t in range(len(Z)):
        Q[t + 1] = c + a * np.outer(Z[t], Z[t]) + b * Q[t]
    return Q

def acorrelacion(Q):
    d = np.sqrt(np.einsum("tii->ti", Q))
    return Q / d[:, :, None] / d[:, None, :]

def nlldcc(p, Z, Qbar):
    a, b = p
    if a <= 1e-6 or b <= 1e-6 or a + b >= 0.999:
        return 1e12
    Rt = acorrelacion(recursionq(Z, a, b, Qbar)[:-1])
    nada, logdet = np.linalg.slogdet(Rt)
    quad = np.einsum("ti,tij,tj->t", Z, np.linalg.inv(Rt), Z)
    return 0.5 * np.sum(logdet + quad - np.einsum("ti,ti->t", Z, Z))

iniciomotor = time.time()
PARAMSDCC = []
# se reestima cada año con datos pasados
for j, a in enumerate(CORTES):
    finb = CORTES[j + 1] - 1 if j + 1 < len(CORTES) else TT
    filas = slice(a - 1, finb)
    Zin = np.empty((a, N))
    for i, c in enumerate(RIESGOSOS):
        res = arch.arch_model(X[c], mean="Constant", vol="GARCH", p=1, o=1, q=1, dist="t").fit(last_obs=a, disp="off")
        f = res.forecast(horizon=H, start=a - 1, reindex=False)
        v = f.variance.values[: finb - a + 1]
        S1[filas, i], S21[filas, i] = v[:, 0], v.sum(axis=1)
        MU[filas, i], NU[filas, i] = f.mean.values[: finb - a + 1, 0], res.params["nu"]
        Zin[:, i] = np.asarray(res.std_resid)[:a]
    Zin = np.nan_to_num(Zin)
    Qbar = Zin.T @ Zin / len(Zin)
    opt = min((optimize.minimize(nlldcc, x0=x0, args=(Zin, Qbar), method="Nelder-Mead", options=dict(xatol=1e-4, fatol=1e-3, maxiter=400))
               for x0 in ([0.03, 0.94], [0.01, 0.98], [0.06, 0.88])), key=lambda r: r.fun)
    ad, bd = opt.x
    PARAMSDCC.append({"Reestimacion": FECHAS[a].strftime("%d/%m/%Y"), "α DCC": ad, "β DCC": bd, "ν mediana": float(np.median(NU[a - 1]))})
    Zoos = (X.values[a:finb] - MU[a - 1:finb - 1]) / np.sqrt(S1[a - 1:finb - 1])
    Q = recursionq(np.vstack([Zin, Zoos]), ad, bd, Qbar)
    RC[filas] = acorrelacion(Q[a:finb + 1])

def cov(o, h=21):
    s = np.sqrt((S21 if h == 21 else S1)[o]) / 100
    return RC[o] * np.outer(s, s)

print(f"Motor de riesgo: {len(CORTES)} reestimaciones anuales en {time.time() - iniciomotor:.0f} s · pronosticos desde el {FECHAS[k0]:%d/%m/%Y}")
mostrartabla(tabla(pd.DataFrame(PARAMSDCC), "Parametros DCC por reestimacion", "α mide la reaccion de las correlaciones a los choques; β, su persistencia")
              .fmt_number(columns=["α DCC", "β DCC"], decimals=4).fmt_number(columns=["ν mediana"], decimals=1), "dcc")

Motor de riesgo: 15 reestimaciones anuales en 187 s · pronosticos desde el 03/01/2012


GT(_tbl_data=   Reestimacion     α DCC     β DCC  ν mediana
0    03/01/2012  0.034361  0.933335   8.440232
1    04/01/2013  0.031808  0.938605   8.533702
2    06/11/2014  0.064837  0.606840   6.659795
3    06/11/2015  0.063244  0.635864   7.224547
4    07/11/2016  0.013453  0.970946   7.825136
5    07/11/2017  0.012675  0.981336   7.280197
6    07/11/2018  0.013111  0.981114   7.283681
7    08/11/2019  0.014504  0.979722   7.146041
8    09/11/2020  0.018990  0.972769   7.118396
9    09/11/2021  0.017760  0.974545   7.120765
10   09/11/2022  0.018061  0.975189   7.483033
11   10/11/2023  0.017786  0.975865   7.303296
12   12/11/2024  0.020014  0.973073   7.330687
13   14/11/2025  0.019472  0.973802   7.108874
14   17/11/2026  0.020283  0.972968   7.027787, _body=<great_tables._gt_data.Body object at 0x000001B4B272AF30>, _boxhead=Boxhead([ColInfo(var='Reestimacion', type=<ColInfoTypeEnum.default: 1>, column_label='Reestimacion', column_align='right', column_width=None, stub_level=None), ColInfo(var='α DCC', type=<ColInfoTypeEnum.default: 1>, column_label='α DCC', column_align='right', column_width=None, stub_level=None), ColInfo(var='β DCC', type=<ColInfoTypeEnum.default: 1>, column_label='β DCC', column_align='right', column_width=None, stub_level=None), ColInfo(var='ν mediana', type=<ColInfoTypeEnum.default: 1>, column_label='ν mediana', column_align='right', column_width=None, stub_level=None)]), _stub=<great_tables._gt_data.Stub object at 0x000001B4B2476990>, _spanners=Spanners([]), _heading=Heading(title='Parametros DCC por reestimacion', subtitle='α mide la reaccion de las correlaciones a los choques; β, su persistencia', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x000001B4B27402F0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x000001B4B2743D10>, _source_notes=['jlsd'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x000001B4B27429F0>, _formats=[<great_tables._gt_data.FormatInfo object at 0x000001B4B272A540>, <great_tables._gt_data.FormatInfo object at 0x000001B4B274E8D0>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='100%'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#1a1a19'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['Times New Roman', 'Times', 'serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='14px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#f0efec'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#383835'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_bord

## 4. Estrategias

Ninguna intenta adivinar rendimientos, todas usan solo el riesgo.

60/40 con freno de riesgo: el 60/40 normal pero baja acciones cuando el riesgo esta alto (75% en tension y 50% en estres). Control de volatilidad: reparte el riesgo parejo y ajusta lo invertido para quedar en VOLOBJETIVO, lo demas va a CETES; la version con bandas solo opera si un activo se movio mas que BANDA. Paridad de riesgo (DCC e historica), Minima varianza, 1/N, 60/40 Mexico, Solo IPC y Solo CETES quedan como comparacion.

Se rebalancea cada fin de mes cobrando comision mas IVA.

In [7]:
# optimizadores
def wminvar(S, wmax):
    n = len(S)
    r = optimize.minimize(lambda w: w @ S @ w, np.full(n, 1 / n), jac=lambda w: 2 * S @ w, bounds=[(0, wmax)] * n,
                          constraints=({"type": "eq", "fun": lambda w: w.sum() - 1},), method="SLSQP", options=dict(ftol=1e-14, maxiter=500))
    w = np.clip(r.x, 0, None)
    return w / w.sum()

def wparidad(S):
    Sn = S / np.mean(np.diag(S))
    n = len(Sn)
    r = optimize.minimize(lambda y: 0.5 * y @ Sn @ y - np.log(y).sum() / n, 1 / np.sqrt(np.diag(Sn)), jac=lambda y: Sn @ y - 1 / (n * y),
                          bounds=[(1e-10, None)] * n, method="L-BFGS-B", options=dict(maxiter=1000))
    return r.x / r.x.sum()

def fijo(**pesos):
    return np.array([pesos.get(k, 0.0) for k in ACT])

# percentil de riesgo para el freno
WEQ = np.full(N, 1 / N)
VOL1N = np.full(TT, np.nan)
for t in range(k0 - 1, TT):
    VOL1N[t] = np.sqrt(WEQ @ cov(t) @ WEQ * 12)

def percentilriesgo(o):
    h = VOL1N[k0 - 1:o + 1]
    return (h < VOL1N[o]).mean() if len(h) >= 63 else 0.0

def freno(o):
    p = percentilriesgo(o)
    return 0.5 if p >= 0.90 else 0.75 if p >= 0.70 else 1.0

# pesos de cada estratejia en una fecha
def pesosen(o):
    S = cov(o)
    f = freno(o)
    Sh = R[RIESGOSOS].iloc[max(0, o - 251):o + 1].cov().values * 21
    rp = wparidad(S)
    vol = np.sqrt(rp @ S @ rp * 12)
    k = min(1.0, VOLOBJETIVO / vol)
    return {
        "60/40 con freno de riesgo": fijo(IVV=0.3 * f, NAFTRAC=0.3 * f, CETES=1 - 0.6 * f),
        "Control de volatilidad (bandas)": np.r_[k * rp, 1 - k],
        "Control de volatilidad": np.r_[k * rp, 1 - k],
        "Paridad de riesgo (DCC)": np.r_[rp, 0.0],
        "Minima varianza (DCC)": np.r_[wminvar(S, PESOMAX), 0.0],
        "Paridad de riesgo (historica)": np.r_[wparidad(Sh), 0.0],
        "1/N": np.r_[np.full(N, 1 / N), 0.0],
        "60/40 Mexico": fijo(IVV=0.3, NAFTRAC=0.3, CETES=0.4),
        "Solo IPC": fijo(NAFTRAC=1.0),
        "Solo CETES": fijo(CETES=1.0),
    }

FINMES = [o for o in range(k0 - 1, TT - 1) if FECHAS[o].month != FECHAS[o + 1].month]
PESOS = {o: pesosen(o) for o in FINMES}
PESOSHOY = pesosen(TT - 1)

Rm = R[ACT].values
def aplicarbandas(objetivo, actual, banda):
    w = actual.copy()
    mover = np.abs(objetivo[:N] - actual[:N]) > banda
    w[:N][mover] = objetivo[:N][mover]
    if w[:N].sum() > 1:
        w[:N] = w[:N] / w[:N].sum()
    w[N] = 1 - w[:N].sum()
    return w

# backtest con comisiones
def simular(nombre):
    origenes = {o: PESOS[o][nombre] for o in FINMES}
    banda = BANDA if nombre in CONBANDAS else None
    primero = FINMES[0]
    valor, w = 1.0, fijo(CETES=1.0)
    v = np.full(TT, np.nan); v[primero] = 1.0
    W = np.full((TT, len(ACT)), np.nan)
    giro, costo = np.zeros(TT), np.zeros(TT)
    for t in range(primero + 1, TT):
        if t - 1 in origenes:
            objetivo = origenes[t - 1]
            if banda is not None and w[:N].sum() > 0:
                objetivo = aplicarbandas(objetivo, w, banda)
            delta = np.abs(objetivo - w)
            costo[t] = (delta * COSTOVEC).sum()
            giro[t] = delta[:N].sum() / 2
            valor *= 1 - costo[t]
            w = objetivo.copy()
        W[t] = w
        rp = w @ Rm[t]
        valor *= 1 + rp
        w = w * (1 + Rm[t]) / (1 + rp)
        v[t] = valor
    idx = FECHAS[primero:]
    return pd.Series(v[primero:], index=idx), pd.DataFrame(W[primero:], index=idx, columns=ACT), pd.Series(giro[primero:], index=idx), pd.Series(costo[primero:], index=idx)

SIM = {e: simular(e) for e in ESTRATEGIAS}
VALOR = pd.DataFrame({e: SIM[e][0] for e in ESTRATEGIAS})
RET = VALOR.pct_change().dropna()
RF = R["CETES"].loc[RET.index]
print(f"Backtest: {len(FINMES)} rebalanceos mensuales del {FECHAS[FINMES[0]]:%d/%m/%Y} al {FECHAS[FINMES[-1]]:%d/%m/%Y} · costo por operacion {100 * COSTO:.3f}%")

Backtest: 172 rebalanceos mensuales del 30/12/2011 al 30/10/2026 · costo por operacion 0.290%


In [8]:
# prueba de sharpe de ledoit wolf
def ledoitwolf(r1, r2):
    y = np.column_stack([r1, r2, r1 ** 2, r2 ** 2])
    mu = y.mean(0)
    m1, m2, g1, g2 = mu
    s1, s2 = g1 - m1 ** 2, g2 - m2 ** 2
    if s1 <= 0 or s2 <= 0:
        return np.nan
    grad = np.array([g1 / s1 ** 1.5, -g2 / s2 ** 1.5, -0.5 * m1 / s1 ** 1.5, 0.5 * m2 / s2 ** 1.5])
    u = y - mu
    L = int(np.floor(4 * (len(u) / 100) ** (2 / 9)))
    psi = u.T @ u / len(u)
    for l in range(1, L + 1):
        g = u[l:].T @ u[:-l] / len(u)
        psi += (1 - l / (L + 1)) * (g + g.T)
    se = np.sqrt(grad @ psi @ grad / len(u))
    return 2 * stats.norm.sf(abs((m1 / np.sqrt(s1) - m2 / np.sqrt(s2)) / se))

def metricas(e):
    r, ex = RET[e], RET[e] - RF
    v = VALOR[e]
    anios = (v.index[-1] - v.index[0]).days / 365.25
    cagr = v.iloc[-1] ** (1 / anios) - 1
    dd = (v / v.cummax() - 1).min()
    mens = v.resample("ME").last().pct_change().dropna()
    sd = ex.std()
    neg = np.sqrt(np.mean(np.minimum(ex, 0) ** 2))
    return {
        "Estrategia": e, "Rend. anual": cagr, "Volatilidad": r.std() * np.sqrt(252),
        "Sharpe": ex.mean() / sd * np.sqrt(252) if sd > 1e-12 else np.nan,
        "Sortino": ex.mean() / neg * np.sqrt(252) if neg > 1e-12 else np.nan,
        "Max. caida": dd, "Calmar": cagr / abs(dd) if dd < -1e-9 else np.nan,
        "Peor mes": mens.min(), "Meses positivos": (mens > 0).mean(),
        "Rotacion anual": SIM[e][2].sum() / anios, "Costos": 1 - (1 - SIM[e][3]).prod(),
        "p Sharpe vs 60/40": ledoitwolf(ex.values, (RET["60/40 Mexico"] - RF).values) if e not in ("60/40 Mexico", "Solo CETES") else np.nan,
        f"Valor final de {mxn(CAPITALMXN)}": CAPITALMXN * v.iloc[-1],
    }

MET = pd.DataFrame([metricas(e) for e in ESTRATEGIAS])
MET.to_csv(DIROUT / "metricas backtest.csv", index=False)
AUTO = ESTRATEGIA == "auto"
if AUTO:
    ESTRATEGIA = MET.set_index("Estrategia")["Sharpe"].drop("Solo CETES").idxmax()
colvalor = MET.columns[-1]
g = (tabla(MET, "Resultados fuera de muestra, netos de comisiones",
           f"Del {VALOR.index[0]:%d/%m/%Y} al {VALOR.index[-1]:%d/%m/%Y} · Sharpe sobre CETES · p de Ledoit Wolf (2008) para la diferencia de Sharpe contra 60/40")
     .fmt_percent(columns=["Rend. anual", "Volatilidad", "Max. caida", "Peor mes", "Meses positivos", "Costos"], decimals=1)
     .fmt_number(columns=["Sharpe", "Sortino", "Calmar"], decimals=2).fmt_percent(columns=["Rotacion anual"], decimals=0)
     .fmt_number(columns=["p Sharpe vs 60/40"], decimals=3).fmt_number(columns=[colvalor], decimals=0, pattern="${x}")
     .sub_missing(missing_text="")
     .data_color(columns=["Sharpe"], palette=T["seq"], na_color=T["sup"])
     .tab_style(style=[style.fill(color=T["banda2"]), style.text(weight="bold")], locations=loc.body(rows=[ESTRATEGIAS.index(ESTRATEGIA)])))
mostrartabla(g, "metricas")

m = MET.set_index("Estrategia")
elegida, ref = m.loc[ESTRATEGIA], m.loc["60/40 Mexico"]
p = elegida["p Sharpe vs 60/40"]
display(Markdown(
    f"**{ESTRATEGIA}** rindio **{100 * elegida['Rend. anual']:.1f}%** anual con volatilidad de {100 * elegida['Volatilidad']:.1f}%, caida maxima de "
    f"{100 * elegida['Max. caida']:.1f}% y Sharpe de {elegida['Sharpe']:.2f}."
    + ("" if ESTRATEGIA == "60/40 Mexico" else
       f" El 60/40 rindio {100 * ref['Rend. anual']:.1f}% con caida maxima de {100 * ref['Max. caida']:.1f}% y Sharpe de {ref['Sharpe']:.2f}"
       + (f" (p = {p:.3f}; " + ("la diferencia es estadisticamente significativa al 5%)." if p < 0.05 else "la diferencia no es estadisticamente significativa al 5%).") if pd.notna(p) else "."))
    + f" Entre todas las estrategias, el Sharpe mas alto es el de **{m['Sharpe'].idxmax()}** ({m['Sharpe'].max():.2f}). "
    + (f"Como `ESTRATEGIA = \"auto\"`, se recomienda la de mejor Sharpe fuera de muestra; al elegir entre {len(ESTRATEGIAS) - 1} estrategias con la misma historia, ese Sharpe tiende a sobreestimar el desempeño futuro. " if AUTO else "")
    + f"Las bandas de {BANDA:.0%} llevaron los costos del control de volatilidad de {100 * m.loc['Control de volatilidad', 'Costos']:.1f}% a "
    f"{100 * m.loc['Control de volatilidad (bandas)', 'Costos']:.1f}% (Sharpe {m.loc['Control de volatilidad', 'Sharpe']:.2f} → {m.loc['Control de volatilidad (bandas)', 'Sharpe']:.2f}). "
    f"El freno de riesgo cambio el Sharpe del 60/40 de {ref['Sharpe']:.2f} a {m.loc['60/40 con freno de riesgo', 'Sharpe']:.2f} (p = {m.loc['60/40 con freno de riesgo', 'p Sharpe vs 60/40']:.3f}) "
    f"y su caida maxima de {100 * ref['Max. caida']:.1f}% a {100 * m.loc['60/40 con freno de riesgo', 'Max. caida']:.1f}%. "
    f"Paridad de riesgo con DCC contra la version historica: Sharpe {m.loc['Paridad de riesgo (DCC)', 'Sharpe']:.2f} contra {m.loc['Paridad de riesgo (historica)', 'Sharpe']:.2f}, "
    f"p = {ledoitwolf((RET['Paridad de riesgo (DCC)'] - RF).values, (RET['Paridad de riesgo (historica)'] - RF).values):.3f}."))

GT(_tbl_data=                        Estrategia  Rend. anual  Volatilidad    Sharpe  \
0        60/40 con freno de riesgo     0.081973     0.063889  0.285375   
1  Control de volatilidad (bandas)     0.084723     0.098542  0.240814   
2           Control de volatilidad     0.082047     0.097346  0.215214   
3          Paridad de riesgo (DCC)     0.083677     0.105732  0.221226   
4    Paridad de riesgo (historica)     0.088880     0.105283  0.270449   
5            Minima varianza (DCC)     0.046189     0.089444 -0.175181   
6                              1/N     0.100840     0.122529  0.343562   
7                     60/40 Mexico     0.095710     0.085590  0.389416   
8                         Solo IPC     0.056541     0.156467  0.019397   
9                       Solo CETES     0.065705     0.005751       NaN   

    Sortino  Max. caida    Calmar  Peor mes  Meses positivos  Rotacion anual  \
0  0.405516   -0.083938  0.976591 -0.033217         0.726257        0.531609   
1  0.362016   -0.182140  0.465156 -0.039643         0.597765        0.385618   
2  0.324253   -0.173466  0.472989 -0.039296         0.603352        0.815093   
3  0.335832   -0.180238  0.464259 -0.044641         0.586592        0.737999   
4  0.409739   -0.171207  0.519136 -0.044042         0.614525        0.262937   
5 -0.255264   -0.180605  0.255743 -0.041099         0.553073        1.605137   
6  0.524957   -0.205867  0.489833 -0.052081         0.603352        0.182426   
7  0.564603   -0.117957  0.811396 -0.043487         0.698324        0.097591   
8  0.027126   -0.364407  0.155158 -0.160814         0.547486        0.033577   
9       NaN    0.000000       NaN  0.000000         0.960894        0.000000   

     Costos  p Sharpe vs 60/40  Valor final de $100,000  
0  0.044889           0.085386            323236.659789  
1  0.032767           0.429000            335691.811662  
2  0.067997           0.376530            323569.722350  
3  0.061767           0.390301            330902.885090  
4  0.022460           0.527962            355364.576627  
5  0.129523           0.009615            195891.321162  
6  0.015637           0.812970            418142.034881  
7  0.008395                NaN            390045.132179  
8  0.002900           0.025433            226826.154493  
9  0.000000                NaN            257958.760380  , _body=<great_tables._gt_data.Body object at 0x000001B4B272BCB0>, _boxhead=Boxhead([ColInfo(var='Estrategia', type=<ColInfoTypeEnum.default: 1>, column_label='Estrategia', column_align='left', column_width=None, stub_level=None), ColInfo(var='Rend. anual', type=<ColInfoTypeEnum.default: 1>, column_label='Rend. anual', column_align='right', column_width=None, stub_level=None), ColInfo(var='Volatilidad', type=<ColInfoTypeEnum.default: 1>, column_label='Volatilidad', column_align='right', column_width=None, stub_level=None), ColInfo(var='Sharpe', type=<ColInfoTypeEnum.default: 1>, column_label='Sharpe', column_align='right', column_width=None, stub_level=None), ColInfo(var='Sortino', type=<ColInfoTypeEnum.default: 1>, column_label='Sortino', column_align='right', column_width=None, stub_level=None), ColInfo(var='Max. caida', type=<ColInfoTypeEnum.default: 1>, column_label='Max. caida', column_align='right', column_width=None, stub_level=None), ColInfo(var='Calmar', type=<ColInfoTypeEnum.default: 1>, column_label='Calmar', column_align='right', column_width=None, stub_level=None), ColInfo(var='Peor mes', type=<ColInfoTypeEnum.default: 1>, column_label='Peor mes', column_align='right', column_width=None, stub_level=None), ColInfo(var='Meses positivos', type=<ColInfoTypeEnum.default: 1>, column_label='Meses positivos', column_align='right', column_width=None, stub_level=None), ColInfo(var='Rotacion anual', type=<ColInfoTypeEnum.default: 1>, column_label='Rotacion anual', column_align='right', column_width=None, stub_level=None), ColInfo(var='Costos', type=<ColInfoTypeEnum.default: 1>, column_label='Costos', column_align='right', co

**60/40 Mexico** rindio **9.6%** anual con volatilidad de 8.6%, caida maxima de -11.8% y Sharpe de 0.39. Entre todas las estrategias, el Sharpe mas alto es el de **60/40 Mexico** (0.39). Como `ESTRATEGIA = "auto"`, se recomienda la de mejor Sharpe fuera de muestra; al elegir entre 9 estrategias con la misma historia, ese Sharpe tiende a sobreestimar el desempeño futuro. Las bandas de 5% llevaron los costos del control de volatilidad de 6.8% a 3.3% (Sharpe 0.22 → 0.24). El freno de riesgo cambio el Sharpe del 60/40 de 0.39 a 0.29 (p = 0.085) y su caida maxima de -11.8% a -8.4%. Paridad de riesgo con DCC contra la version historica: Sharpe 0.22 contra 0.27, p = 0.131.

In [9]:
fig = go.Figure()
for e in ESTRATEGIAS:
    s = CAPITALMXN * VALOR[e]
    sel = e == ESTRATEGIA
    fig.add_scatter(x=s.index, y=s, name=e, line=dict(color=COLOREST[e], width=3.2 if sel else 1.6, dash=LINEAEST.get(e, "solid")),
                    opacity=1 if sel else 0.85, hovertemplate="%{y:$,.0f}<extra>" + e + "</extra>")
tv = tickslog(CAPITALMXN * VALOR.min().min(), CAPITALMXN * VALOR.max().max(), CAPITALMXN)
etiquetasderecha(fig, [(f"<b>{e}</b> {mxn(CAPITALMXN * VALOR[e].iloc[-1])}" if e == ESTRATEGIA else f"{e} {mxn(CAPITALMXN * VALOR[e].iloc[-1])}",
                         CAPITALMXN * VALOR[e].iloc[-1], T["txt"] if e == ESTRATEGIA else T["txt2"]) for e in ESTRATEGIAS], tv)
fig.update_yaxes(type="log", tickvals=tv, ticktext=[mxn(v) for v in tv])
fig.update_xaxes(range=[VALOR.index[0], VALOR.index[-1]])
fig.update_layout(hovermode="x unified", margin=dict(r=320, t=170), height=620, legend=dict(y=1.04))
titulo(fig, f"{mxn(CAPITALMXN)} invertidos con cada estrategia", f"Fuera de muestra y netos de comisiones · linea gruesa: {ESTRATEGIA}")
mostrar(fig, "curvascapital")

fig = go.Figure()
for e in dict.fromkeys([ESTRATEGIA, "60/40 con freno de riesgo", "60/40 Mexico", "Solo IPC"]):
    dd = VALOR[e] / VALOR[e].cummax() - 1
    fig.add_scatter(x=dd.index, y=dd, name=e, line=dict(color=COLOREST[e], width=2), fill="tozeroy" if e == ESTRATEGIA else None,
                    fillcolor=T["banda"], hovertemplate="%{y:.1%}<extra>" + e + "</extra>")
fig.update_yaxes(tickformat=".0%")
fig.update_layout(hovermode="x unified", height=400)
titulo(fig, "Que tan hondo cae cada cartera", "Caida desde el maximo previo")
mostrar(fig, "caidas")

def colocaretiquetas(xs, ys, textos, ancho=1100, alto=300):
    x0, x1 = min(xs) - 0.012, max(xs) + 0.012
    y0, y1 = min(ys) - 0.006, max(ys) + 0.006
    px = lambda x, y: ((x - x0) / (x1 - x0) * ancho, (y1 - y) / (y1 - y0) * alto)
    puntos = [px(x, y) for x, y in zip(xs, ys)]
    cajas, salida = [], []
    opciones = [("top center", 0, -16, 0), ("bottom center", 0, 16, 0), ("middle right", 1, 0, 10), ("middle left", -1, 0, 10),
                ("top right", 1, -14, 0), ("top left", -1, -14, 0), ("bottom right", 1, 14, 0), ("bottom left", -1, 14, 0)]
    for (cx, cy), t in zip(puntos, textos):
        w, h = 7.4 * len(t), 16
        elegida = "top center"
        for nombre, lado, dy, extra in opciones:
            mx = cx + lado * (w / 2 + extra)
            caja = (mx - w / 2, cy + dy - h / 2, mx + w / 2, cy + dy + h / 2)
            choca = any(not (caja[2] < c[0] or caja[0] > c[2] or caja[3] < c[1] or caja[1] > c[3]) for c in cajas)
            choca = choca or any(caja[0] - 8 < qx < caja[2] + 8 and caja[1] - 8 < qy < caja[3] + 8 for qx, qy in puntos)
            if not choca:
                elegida = nombre
                cajas.append(caja)
                break
        salida.append(elegida)
    return salida, (x0, x1), (y0, y1)

ordene = [ESTRATEGIA] + [e for e in m["Sharpe"].fillna(-9).sort_values(ascending=False).index if e != ESTRATEGIA]
posiciones, rx, ry = colocaretiquetas([m.loc[e, "Volatilidad"] for e in ordene], [m.loc[e, "Rend. anual"] for e in ordene], ordene)
fig = go.Figure()
for e, posicion in zip(ordene, posiciones):
    sel = e == ESTRATEGIA
    xv, yv = m.loc[e, "Volatilidad"], m.loc[e, "Rend. anual"]
    fig.add_scatter(x=[xv], y=[yv], mode="markers+text", text=[e], textposition=posicion,
                    marker=dict(size=16 if sel else 11, color=T["serie"][0] if sel else T["mudo"], line=dict(color=T["sup"], width=2)),
                    textfont=dict(color=T["txt"] if sel else T["txt2"], size=12), showlegend=False,
                    hovertemplate=f"<b>{e}</b><br>Volatilidad %{{x:.1%}}<br>Rendimiento %{{y:.1%}}<br>Sharpe {m.loc[e, 'Sharpe']:.2f}<extra></extra>")
fig.update_xaxes(range=list(rx))
fig.update_yaxes(range=list(ry))
fig.update_xaxes(tickformat=".0%", title="Volatilidad anual", showgrid=True, gridcolor=T["rejilla"])
fig.update_yaxes(tickformat=".0%", title="Rendimiento anual")
fig.update_layout(height=480)
titulo(fig, "Riesgo contra rendimiento", "Cada punto es una estrategia en todo el periodo fuera de muestra")
mostrar(fig, "riesgorendimiento")

W = SIM[ESTRATEGIA][1].resample("ME").first()
fig = go.Figure()
for k in ACT:
    fig.add_scatter(x=W.index, y=W[k], name=ACTIVOS[k]["nombre"], stackgroup="pesos", mode="lines", line=dict(width=0.6, color=T["sup"]),
                    fillcolor=COLORACT[k], hovertemplate="%{y:.0%}<extra>" + ACTIVOS[k]["nombre"] + "</extra>")
fig.update_yaxes(tickformat=".0%", range=[0, 1])
fig.update_layout(hovermode="x unified", legend=dict(traceorder="normal"))
titulo(fig, f"Como se ha repartido el dinero en {ESTRATEGIA}", "Pesos al inicio de cada mes")
mostrar(fig, "pesoshistoricos")

if ESTRATEGIA.startswith("Control de volatilidad"):
    vr = RET[ESTRATEGIA].rolling(63).std() * np.sqrt(252)
    fig = go.Figure()
    fig.add_scatter(x=vr.index, y=vr, name="Volatilidad realizada (63 dias)", line=dict(color=COLOREST[ESTRATEGIA], width=2), hovertemplate="%{y:.1%}<extra></extra>")
    fig.add_hline(y=VOLOBJETIVO, line=dict(color=T["mudo"], dash="dash", width=1.5), annotation_text=f"Objetivo {VOLOBJETIVO:.0%}",
                  annotation_font_color=T["txt2"], annotation_position="top left")
    fig.update_yaxes(tickformat=".0%")
    fig.update_layout(height=380, showlegend=False)
    titulo(fig, "¿Se cumplio el objetivo de volatilidad?", f"Volatilidad realizada de la cartera contra el objetivo de {VOLOBJETIVO:.0%}")
    mostrar(fig, "volobjetivo")

## 5. Validacion del VaR

Se revisa si el VaR al 99% falla el 1% de las veces (Kupiec) y si las fallas no llegan juntas (Christoffersen).

In [10]:
# backtest del var
def backtestvar(e, alfa=0.01):
    W = SIM[e][1]
    pos = FECHAS.get_indexer(W.index)
    var, real = [], []
    for t, wt in zip(pos, W.values):
        o = t - 1
        if o < k0 - 1 or np.isnan(wt).any() or wt[:N].sum() < 1e-6:
            continue
        wr = wt[:N]
        nu = np.median(NU[o])
        q = stats.t.ppf(alfa, nu) * np.sqrt((nu - 2) / nu)
        s = np.sqrt(wr @ cov(o, 1) @ wr)
        var.append(wr @ MU[o] / 100 + wt[N] * Rm[t, N] + q * s)
        real.append(wt @ Rm[t])
    h = (np.array(real) < np.array(var)).astype(int)
    Tn, x = len(h), h.sum()
    ll = lambda x, T, p: xlogy(T - x, 1 - p) + xlogy(x, p)
    lrpof = -2 * (ll(x, Tn, alfa) - ll(x, Tn, x / Tn))
    h0, h1 = h[:-1], h[1:]
    n00, n01, n10, n11 = (((h0 == a) & (h1 == b)).sum() for a, b in [(0, 0), (0, 1), (1, 0), (1, 1)])
    p01, p11, pp = n01 / max(n00 + n01, 1), n11 / max(n10 + n11, 1), (n01 + n11) / max(Tn - 1, 1)
    lrind = -2 * ((xlogy(n00 + n10, 1 - pp) + xlogy(n01 + n11, pp)) - (xlogy(n00, 1 - p01) + xlogy(n01, p01) + xlogy(n10, 1 - p11) + xlogy(n11, p11)))
    pcc = stats.chi2.sf(lrpof + lrind, 2)
    return {"Estrategia": e, "Dias": Tn, "Violaciones": int(x), "Esperadas": Tn * alfa, "Kupiec p": stats.chi2.sf(lrpof, 1),
            "Christoffersen p": stats.chi2.sf(lrind, 1), "Cobertura cond. p": pcc, "Veredicto": "Confiable" if pcc >= 0.05 else "Subestima o agrupa"}

VARBT = pd.DataFrame([backtestvar(e) for e in dict.fromkeys([ESTRATEGIA, "60/40 Mexico", "Solo IPC"]) if e != "Solo CETES"])
g = (tabla(VARBT, "Validacion del VaR al 99% a un dia", "Fuera de muestra, con los pesos efectivamente mantenidos")
     .fmt_number(columns=["Esperadas"], decimals=1).fmt_number(columns=["Kupiec p", "Christoffersen p", "Cobertura cond. p"], decimals=3))
for i, v in enumerate(VARBT["Veredicto"]):
    g = g.tab_style(style=[style.fill(color=ESTADO["verde"] if v == "Confiable" else ESTADO["rojo"]), style.text(color="#0b0b0b", weight="bold")], locations=loc.body(columns="Veredicto", rows=[i]))
mostrartabla(g, "var")
VAROK = VARBT.set_index("Estrategia").get("Cobertura cond. p", pd.Series(dtype=float)).get(ESTRATEGIA, np.nan)

GT(_tbl_data=     Estrategia  Dias  Violaciones  Esperadas  Kupiec p  Christoffersen p  \
0  60/40 Mexico  3532           41      35.32  0.348944          0.502236   
1      Solo IPC  3532           42      35.32  0.272669          0.527433   

   Cobertura cond. p  Veredicto  
0           0.514917  Confiable  
1           0.448744  Confiable  , _body=<great_tables._gt_data.Body object at 0x000001B4B2761220>, _boxhead=Boxhead([ColInfo(var='Estrategia', type=<ColInfoTypeEnum.default: 1>, column_label='Estrategia', column_align='left', column_width=None, stub_level=None), ColInfo(var='Dias', type=<ColInfoTypeEnum.default: 1>, column_label='Dias', column_align='right', column_width=None, stub_level=None), ColInfo(var='Violaciones', type=<ColInfoTypeEnum.default: 1>, column_label='Violaciones', column_align='right', column_width=None, stub_level=None), ColInfo(var='Esperadas', type=<ColInfoTypeEnum.default: 1>, column_label='Esperadas', column_align='right', column_width=None, stub_level=None), ColInfo(var='Kupiec p', type=<ColInfoTypeEnum.default: 1>, column_label='Kupiec p', column_align='right', column_width=None, stub_level=None), ColInfo(var='Christoffersen p', type=<ColInfoTypeEnum.default: 1>, column_label='Christoffersen p', column_align='right', column_width=None, stub_level=None), ColInfo(var='Cobertura cond. p', type=<ColInfoTypeEnum.default: 1>, column_label='Cobertura cond. p', column_align='right', column_width=None, stub_level=None), ColInfo(var='Veredicto', type=<ColInfoTypeEnum.default: 1>, column_label='Veredicto', column_align='left', column_width=None, stub_level=None)]), _stub=<great_tables._gt_data.Stub object at 0x000001B4B248EC90>, _spanners=Spanners([]), _heading=Heading(title='Validacion del VaR al 99% a un dia', subtitle='Fuera de muestra, con los pesos efectivamente mantenidos', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x000001B4B277BE90>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x000001B4B277BEC0>, _source_notes=['jlsd'], _footnotes=[], _styles=[StyleInfo(locname=LocBody(columns='Veredicto', rows=[0], mask=None), grpname=None, colname='Veredicto', rownum=0, colnum=None, styles=[CellStyleFill(color='#0ca30c'), CellStyleText(color='#0b0b0b', font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocBody(columns='Veredicto', rows=[1], mask=None), grpname=None, colname='Veredicto', rownum=1, colnum=None, styles=[CellStyleFill(color='#0ca30c'), CellStyleText(color='#0b0b0b', font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)])], _locale=<great_tables._gt_data.Locale object at 0x000001B4B277BEF0>, _formats=[<great_tables._gt_data.FormatInfo object at 0x000001B4B249A390>, <great_tables._gt_data.FormatInfo object at 0x000001B4B27F8EF0>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='100%'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#1a1a19'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['Times New Roman', 'Times', 'serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='14px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='no

## 6. Decision de hoy

In [11]:
# ordenes de hoy
o = TT - 1
wobj = PESOSHOY[ESTRATEGIA].copy()
S21h, S1h = cov(o), cov(o, 1)
total = float(CAPITALMXN)
actual = np.array([float(POSICIONESMXN.get(k, 0.0)) for k in ACT])
nueva = actual.sum() == 0
if nueva:
    actual[-1] = total
else:
    actual[-1] += max(total - actual.sum(), 0.0)
    total = actual.sum()
wact = actual / total
if ESTRATEGIA in CONBANDAS and not nueva:
    wobj = aplicarbandas(wobj, wact, BANDA)
objetivo = wobj * total
dif = objetivo - actual
comision = np.abs(dif) * COSTOVEC
accion = np.where(np.abs(dif) < 0.005 * total, "Mantener", np.where(dif > 0, "Comprar", "Vender"))
ORDENES = pd.DataFrame({"Clave": ACT, "Activo": [ACTIVOS[k]["nombre"] for k in ACT], "Ticker": [usados.get(k, ACTIVOS[k]["ticker"]) if k != "CETES" else "CETES 28d" for k in ACT],
                        "Peso actual": wact, "Peso objetivo": wobj, "Monto actual": actual, "Monto objetivo": objetivo,
                        "Accion": accion, "Monto a operar": np.abs(dif), "Comision estimada": comision})
desv = np.abs(wobj - wact).max()
if nueva:
    DECISION, colordec = "Cartera nueva: invertir segun la tabla", ESTADO["verde"]
elif desv > UMBRALREBALANCEO:
    DECISION, colordec = f"Rebalancear: desviacion maxima de {desv:.1%}", ESTADO["amarillo"]
else:
    DECISION, colordec = f"Mantener: desviacion maxima de {desv:.1%}, menor al umbral de {UMBRALREBALANCEO:.0%}", ESTADO["verde"]

wr = wobj[:N]
volanual = np.sqrt(wr @ S21h @ wr * 12)
nuhoy = np.median(NU[o])
q99 = -stats.t.ppf(0.01, nuhoy) * np.sqrt((nuhoy - 2) / nuhoy)
var1 = total * q99 * np.sqrt(wr @ S1h @ wr)
var21 = total * stats.norm.ppf(0.99) * np.sqrt(wr @ S21h @ wr)
pct = percentilriesgo(o)
semaforo = ("Calma", ESTADO["verde"]) if pct < 0.70 else ("Tension", ESTADO["amarillo"]) if pct < 0.90 else ("Estres", ESTADO["rojo"])
wmes = PESOS[FINMES[-1]][ESTRATEGIA]
proximo = (pd.Timestamp(FECHAS[-1]) + pd.offsets.BMonthEnd(0)).date()

tarjetas([
    ("Decision", DECISION.split(":")[0], colordec),
    ("Invertido con riesgo", f"{wr.sum():.0%}"),
    ("En CETES", f"{wobj[-1]:.0%}"),
    ("Perdida en un mal dia", mxn(var1)),
    ("Perdida en un mal mes", mxn(var21)),
    ("Clima de mercado", semaforo[0], semaforo[1]),
])

totalfila = pd.DataFrame([{"Activo": "Total", "Peso actual": wact.sum(), "Peso objetivo": wobj.sum(), "Monto actual": actual.sum(), "Monto objetivo": objetivo.sum(),
                            "Monto a operar": np.abs(dif).sum(), "Comision estimada": comision.sum()}])
vista = pd.concat([ORDENES.drop(columns="Clave"), totalfila], ignore_index=True)
cambios = [f"{k} {100 * (a - b):+.1f} pp" for k, a, b in zip(ACT, wobj, wmes) if abs(a - b) >= 0.005]
g = (tabla(vista, f"Ordenes para hoy · {ESTRATEGIA}",
           f"Capital {mxn(total)}")
     .fmt_percent(columns=["Peso actual", "Peso objetivo"], decimals=1)
     .fmt_number(columns=["Monto actual", "Monto objetivo", "Monto a operar", "Comision estimada"], decimals=0, pattern="${x}")
     .sub_missing(missing_text="")
     .tab_style(style=[style.text(weight="bold"), style.borders(sides="top", color=T["eje"], weight="2px")], locations=loc.body(rows=[len(ACT)]))
     .tab_source_note((f"Cambio contra el rebalanceo del {FECHAS[FINMES[-1]]:%d/%m/%Y}: " + ", ".join(cambios) + ".") if cambios
                      else f"Sin cambios de peso contra el rebalanceo del {FECHAS[FINMES[-1]]:%d/%m/%Y}."))
for i, a in enumerate(accion):
    if a != "Mantener":
        g = g.tab_style(style=[style.fill(color=ESTADO["verde"] if a == "Comprar" else ESTADO["naranja"]), style.text(color="#0b0b0b", weight="bold")], locations=loc.body(columns="Accion", rows=[i]))
mostrartabla(g, "ordenes")
ORDENES.to_csv(DIROUT / "ordenes hoy.csv", index=False)

GT(_tbl_data=                         Activo           Ticker  Peso actual  Peso objetivo  \
0                       S&P 500              IVV          0.0            0.3   
1                    Nasdaq 100              QQQ          0.0            0.0   
2                    IPC Mexico  NAFTRACISHRS.MX          0.0            0.3   
3                           Oro              GLD          0.0            0.0   
4  Bonos del Tesoro 7 a 10 años              IEF          0.0            0.0   
5            Dolares en T Bills              BIL          0.0            0.0   
6                 CETES 28 dias        CETES 28d          1.0            0.4   
7                         Total              NaN          1.0            1.0   

   Monto actual  Monto objetivo    Accion  Monto a operar  Comision estimada  
0           0.0         30000.0   Comprar         30000.0               87.0  
1           0.0             0.0  Mantener             0.0                0.0  
2           0.0         30000.0   Comprar         30000.0               87.0  
3           0.0             0.0  Mantener             0.0                0.0  
4           0.0             0.0  Mantener             0.0                0.0  
5           0.0             0.0  Mantener             0.0                0.0  
6      100000.0         40000.0    Vender         60000.0                0.0  
7      100000.0        100000.0       NaN        120000.0              174.0  , _body=<great_tables._gt_data.Body object at 0x000001B4B27F9460>, _boxhead=Boxhead([ColInfo(var='Activo', type=<ColInfoTypeEnum.default: 1>, column_label='Activo', column_align='left', column_width=None, stub_level=None), ColInfo(var='Ticker', type=<ColInfoTypeEnum.default: 1>, column_label='Ticker', column_align='left', column_width=None, stub_level=None), ColInfo(var='Peso actual', type=<ColInfoTypeEnum.default: 1>, column_label='Peso actual', column_align='right', column_width=None, stub_level=None), ColInfo(var='Peso objetivo', type=<ColInfoTypeEnum.default: 1>, column_label='Peso objetivo', column_align='right', column_width=None, stub_level=None), ColInfo(var='Monto actual', type=<ColInfoTypeEnum.default: 1>, column_label='Monto actual', column_align='right', column_width=None, stub_level=None), ColInfo(var='Monto objetivo', type=<ColInfoTypeEnum.default: 1>, column_label='Monto objetivo', column_align='right', column_width=None, stub_level=None), ColInfo(var='Accion', type=<ColInfoTypeEnum.default: 1>, column_label='Accion', column_align='left', column_width=None, stub_level=None), ColInfo(var='Monto a operar', type=<ColInfoTypeEnum.default: 1>, column_label='Monto a operar', column_align='right', column_width=None, stub_level=None), ColInfo(var='Comision estimada', type=<ColInfoTypeEnum.default: 1>, column_label='Comision estimada', column_align='right', column_width=None, stub_level=None)]), _stub=<great_tables._gt_data.Stub object at 0x000001B4B2729010>, _spanners=Spanners([]), _heading=Heading(title='Ordenes para hoy · 60/40 Mexico', subtitle='Capital $100,000', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x000001B4B27F9C40>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x000001B4B27F9D00>, _source_notes=['Sin cambios de peso contra el rebalanceo del 30/10/2026.', 'jlsd'], _footnotes=[], _styles=[StyleInfo(locname=LocBody(columns=None, rows=[7], mask=None), grpname=None, colname='Activo', rownum=7, colnum=None, styles=[CellStyleText(color=None, font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None), CellStyleBorders(sides='top', color='#383835', style='solid', weight='2px')]), StyleInfo(locname=LocBody(columns=None, rows=[7], mask=None), grpname=None, colname='Ticker', rownum=7, colnum=None, styles=[CellStyleText(color=None, font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespac

In [12]:
# riesgo por activo
Sw = S21h @ wr
contrib = wr * Sw / (wr @ Sw) if wr.sum() > 0 else np.zeros(N)
nombres = [ACTIVOS[k]["nombre"] for k in RIESGOSOS]
fig = go.Figure()
fig.add_bar(y=nombres, x=wr, name="Peso en la cartera", orientation="h", marker=dict(color=T["serie"][0], cornerradius=4),
            hovertemplate="%{x:.1%}<extra>Peso</extra>")
fig.add_bar(y=nombres, x=contrib * wr.sum(), name="Aporte al riesgo", orientation="h", marker=dict(color=T["serie"][1], cornerradius=4),
            hovertemplate="%{x:.1%}<extra>Aporte al riesgo</extra>")
fig.update_xaxes(tickformat=".0%", showgrid=True, gridcolor=T["rejilla"])
fig.update_yaxes(autorange="reversed", showgrid=False)
fig.update_layout(barmode="group", bargap=0.3, bargroupgap=0.08, height=430, margin=dict(l=200))
titulo(fig, "Donde esta el dinero y donde esta el riesgo", "Aporte de cada activo al riesgo total, escalado a la parte invertida con riesgo")
mostrar(fig, "riesgoporactivo")

volhist = pd.DataFrame(np.sqrt(S21[k0 - 1:] * 12) / 100, index=FECHAS[k0 - 1:], columns=RIESGOSOS)
fig = go.Figure()
for i, k in enumerate(RIESGOSOS):
    s = volhist[k]
    p05, p50, p95, hoy = s.quantile(0.05), s.median(), s.quantile(0.95), s.iloc[-1]
    fig.add_scatter(x=[p05, p95], y=[nombres[i]] * 2, mode="lines", line=dict(color=T["rejilla"], width=12), showlegend=False, hoverinfo="skip")
    fig.add_scatter(x=[p50], y=[nombres[i]], mode="markers", marker=dict(symbol="diamond", size=11, line=dict(color=T["mudo"], width=2)),
                    showlegend=i == 0, name="Mediana historica", hovertemplate="Mediana %{x:.1%}<extra></extra>")
    fig.add_scatter(x=[hoy], y=[nombres[i]], mode="markers", marker=dict(size=14, color=COLORACT[k], line=dict(color=T["sup"], width=2)),
                    showlegend=False, hovertemplate=f"Hoy %{{x:.1%}} · percentil {(s < hoy).mean():.0%}<extra>{nombres[i]}</extra>")
fig.update_xaxes(tickformat=".0%", showgrid=True, gridcolor=T["rejilla"], title="Volatilidad anual pronosticada para el proximo mes")
fig.update_yaxes(autorange="reversed", showgrid=False)
fig.update_layout(height=420, margin=dict(l=200))
titulo(fig, "Termometro de riesgo por activo", "Barra: rango historico entre los percentiles 5 y 95 · punto: pronostico de hoy")
mostrar(fig, "termometro")

rho = pd.DataFrame(RC[o], index=nombres, columns=nombres)
fig = go.Figure(go.Heatmap(z=rho.values, x=nombres, y=nombres, zmin=-1, zmax=1, colorscale=[[0, T["div"][0]], [0.5, T["div"][1]], [1, T["div"][2]]],
                           text=np.round(rho.values, 2), texttemplate="%{text:.2f}", textfont=dict(size=12),
                           xgap=2, ygap=2, hovertemplate="%{y} · %{x}: %{z:.2f}<extra></extra>", colorbar=dict(thickness=12, outlinewidth=0)))
fig.update_yaxes(autorange="reversed", showgrid=False)
fig.update_xaxes(showgrid=False, ticks="")
fig.update_layout(height=540, margin=dict(l=200))
titulo(fig, "Correlaciones pronosticadas para mañana", "Modelo DCC · azul: se mueven en sentido contrario, rojo: se mueven juntos")
mostrar(fig, "correlaciones")

## 7. Escenarios a un año (R)

El script de R simula 10,000 trayectorias de un año con bootstrap estacionario en bloques de un mes usando toda la historia. Todas las carteras usan los mismos ecenarios.

In [13]:
# busca R y corre los escenarios
def ubicarrscript():
    r = shutil.which("Rscript")
    if r:
        return r
    raices = [Path("C:/Program Files/R"), Path.home() / "AppData/Local/Programs/R", Path("C:/R")]
    version = lambda p: [int(x) for x in re.findall(r"\d+", next(s for s in p.parts if s.startswith("R-")))]
    for patron in ("R-*/bin/x64/Rscript.exe", "R-*/bin/Rscript.exe"):
        cands = sorted((p for r in raices if r.exists() for p in r.glob(patron)), key=version)
        if cands:
            return str(cands[-1])
    return None

carteras = list(dict.fromkeys([ESTRATEGIA, "60/40 Mexico", "Solo IPC", "Solo CETES"]))
entrada = pd.DataFrame({c: R[ACT].values @ PESOSHOY[c] for c in carteras}, index=FECHAS)
entrada.to_csv(DIRPROC / "escenarios entrada.csv", index_label="Fecha")
for f in ("escenarios bandas.csv", "escenarios resumen.csv"):
    (DIROUT / f).unlink(missing_ok=True)
RSCRIPT = ubicarrscript()
if RSCRIPT:
    proc = subprocess.run([RSCRIPT, str(RAIZ / "R" / "escenarios.R"), str(DIRPROC / "escenarios entrada.csv"), str(DIROUT), str(total), "10000", "252", "21"],
                          capture_output=True, text=True)
    print(proc.stdout.strip() or proc.stderr[-1500:])
else:
    print("No se encontro Rscript, instala R desde la pagina de CRAN (solo usa R base)")
HAYR = (DIROUT / "escenarios resumen.csv").exists()

Escenarios listos: 3 carteras, 10000 simulaciones de 252 dias


In [14]:
if HAYR:
    bandas = pd.read_csv(DIROUT / "escenarios bandas.csv")
    ESC = pd.read_csv(DIROUT / "escenarios resumen.csv")
    hoy = pd.Timestamp(FECHAS[-1])
    fig = go.Figure()
    b = bandas[bandas["cartera"] == ESTRATEGIA]
    x = pd.bdate_range(hoy, periods=len(b) + 1)[1:]
    fig.add_scatter(x=x, y=b["p95"], line=dict(width=0), showlegend=False, hoverinfo="skip")
    fig.add_scatter(x=x, y=b["p05"], fill="tonexty", fillcolor=T["banda"], line=dict(width=0), name="90% de los escenarios", hoverinfo="skip")
    fig.add_scatter(x=x, y=b["p75"], line=dict(width=0), showlegend=False, hoverinfo="skip")
    fig.add_scatter(x=x, y=b["p25"], fill="tonexty", fillcolor=T["banda2"], line=dict(width=0), name="50% de los escenarios", hoverinfo="skip")
    fig.add_scatter(x=x, y=b["p50"], name=f"Mediana · {ESTRATEGIA}", line=dict(color=COLOREST[ESTRATEGIA], width=2.5), hovertemplate="%{y:$,.0f}<extra>Mediana</extra>")
    for c in [k for k in ["60/40 Mexico", "Solo IPC"] if k != ESTRATEGIA]:
        bc = bandas[bandas["cartera"] == c]
        fig.add_scatter(x=x, y=bc["p50"], name=f"Mediana · {c}", line=dict(color=COLOREST[c], width=1.6, dash="dash"), hovertemplate="%{y:$,.0f}<extra>" + c + "</extra>")
    fig.add_hline(y=total, line=dict(color=T["mudo"], width=1))
    fig.update_yaxes(tickprefix="$", tickformat=",.0f")
    fig.update_layout(hovermode="x unified")
    titulo(fig, f"Que podria pasar con {mxn(total)} en un año", "10,000 escenarios con bootstrap estacionario")
    mostrar(fig, "escenarios")

    esc = ESC.rename(columns={"cartera": "Cartera", "mediana": "Valor mediano", "p05": "Escenario malo (5%)", "p95": "Escenario bueno (95%)",
                              "probperdida": "Prob. de perder", "probcetes": "Prob. de ganarle a CETES",
                              "caidamediana": "Caida tipica en el año", "caidamala": "Caida en un año malo"})
    g = (tabla(esc, "Escenarios a 12 meses", f"Capital inicial {mxn(total)} · mismos escenarios para todas las carteras")
         .fmt_number(columns=["Valor mediano", "Escenario malo (5%)", "Escenario bueno (95%)"], decimals=0, pattern="${x}")
         .fmt_percent(columns=["Prob. de perder", "Prob. de ganarle a CETES", "Caida tipica en el año", "Caida en un año malo"], decimals=1)
         .sub_missing(missing_text=""))
    mostrartabla(g, "escenarios")
else:
    ESC = None
    print("Sin escenarios en esta corrida")

GT(_tbl_data=        Cartera  Valor mediano  Escenario malo (5%)  Escenario bueno (95%)  \
0  60/40 Mexico  109393.065311         96118.927222          123571.937992   
1      Solo IPC  106824.107045         79006.095814          139521.252723   
2    Solo CETES  106613.797074        105078.338259          108633.801639   

   Prob. de perder  Prob. de ganarle a CETES  Caida tipica en el año  \
0           0.1189                    0.6346               -0.060487   
1           0.3472                    0.5037               -0.142962   
2           0.0000                       NaN                0.000000   

   Caida en un año malo  
0             -0.137033  
1             -0.326274  
2              0.000000  , _body=<great_tables._gt_data.Body object at 0x000001B4B2762420>, _boxhead=Boxhead([ColInfo(var='Cartera', type=<ColInfoTypeEnum.default: 1>, column_label='Cartera', column_align='left', column_width=None, stub_level=None), ColInfo(var='Valor mediano', type=<ColInfoTypeEnum.default: 1>, column_label='Valor mediano', column_align='right', column_width=None, stub_level=None), ColInfo(var='Escenario malo (5%)', type=<ColInfoTypeEnum.default: 1>, column_label='Escenario malo (5%)', column_align='right', column_width=None, stub_level=None), ColInfo(var='Escenario bueno (95%)', type=<ColInfoTypeEnum.default: 1>, column_label='Escenario bueno (95%)', column_align='right', column_width=None, stub_level=None), ColInfo(var='Prob. de perder', type=<ColInfoTypeEnum.default: 1>, column_label='Prob. de perder', column_align='right', column_width=None, stub_level=None), ColInfo(var='Prob. de ganarle a CETES', type=<ColInfoTypeEnum.default: 1>, column_label='Prob. de ganarle a CETES', column_align='right', column_width=None, stub_level=None), ColInfo(var='Caida tipica en el año', type=<ColInfoTypeEnum.default: 1>, column_label='Caida tipica en el año', column_align='right', column_width=None, stub_level=None), ColInfo(var='Caida en un año malo', type=<ColInfoTypeEnum.default: 1>, column_label='Caida en un año malo', column_align='right', column_width=None, stub_level=None)]), _stub=<great_tables._gt_data.Stub object at 0x000001B4B1497020>, _spanners=Spanners([]), _heading=Heading(title='Escenarios a 12 meses', subtitle='Capital inicial $100,000 · mismos escenarios para todas las carteras', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x000001B4B248E030>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x000001B4B7C8A1B0>, _source_notes=['jlsd'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x000001B4B7C89610>, _formats=[<great_tables._gt_data.FormatInfo object at 0x000001B4B2475400>, <great_tables._gt_data.FormatInfo object at 0x000001B4B7CF46E0>], _substitutions=[<great_tables._gt_data.FormatInfo object at 0x000001B4B7CF47A0>], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='100%'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#1a1a19'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['Times New Roman', 'Times', 'serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='14px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo

## 8. Reporte

Se guarda el Reporte final con la fecha, las ordenes y el historial de cada recomendacion en la carpeta resultados.

In [15]:
# historial y reporte final
GENERADO = datetime.now()
registro = {
    "fechadatos": FECHAS[-1].strftime("%d/%m/%Y"), "generado": f"{GENERADO:%d/%m/%Y %H:%M}", "estrategia": ESTRATEGIA, "decision": DECISION,
    "capital": total, "pesosobjetivo": dict(zip(ACT, np.round(wobj, 4))), "volpronosticada": volanual, "var1d99": var1, "var21d99": var21,
    "clima": semaforo[0], "percentilriesgo": pct, "precios": ORIGENP, "cetes": ORIGENC,
    "versiones": {"python": platform.python_version(), "pandas": pd.__version__, "arch": arch.__version__, "plotly": plotly.__version__},
}
(DIRHIST / f"{FECHAS[-1]:%d %m %Y}.json").write_text(json.dumps(registro, indent=1, ensure_ascii=False, default=float), encoding="utf8")
fila = pd.DataFrame([{"fechadatos": registro["fechadatos"], "generado": registro["generado"], "estrategia": ESTRATEGIA, "decision": DECISION,
                      "volpronosticada": volanual, "var1d99": var1, "clima": semaforo[0], **{f"w{k}": v for k, v in zip(ACT, wobj)}}])
histpath = DIROUT / "historial decisiones.csv"
hist = pd.concat([pd.read_csv(histpath), fila]) if histpath.exists() else fila
hist.drop_duplicates(subset=["fechadatos", "estrategia"], keep="last").to_csv(histpath, index=False)

def htmlfig(fig):
    return fig.to_html(full_html=False, include_plotlyjs=False, config={"displaylogo": False, "responsive": True})

seccion = lambda t: f"<h2 style='color:{T['txt']};font-weight:600;margin:36px 0 10px'>{t}</h2>"
bloques = {n: htmlfig(f) for n, f in FIGURAS}
tablashtml = {n: g.as_raw_html() for n, g in TABLAS}
orden = [("Decision de hoy", TARJETAS[-1:] + [tablashtml.get("ordenes", "")] + [bloques.get(k, "") for k in ("riesgoporactivo", "termometro", "correlaciones")]),
         ("Escenarios a un año", [bloques.get("escenarios", ""), tablashtml.get("escenarios", "")]),
         ("Resultados historicos fuera de muestra", [tablashtml.get("metricas", "")] + [bloques.get(k, "") for k in ("curvascapital", "caidas", "riesgorendimiento", "pesoshistoricos", "volobjetivo")] + [tablashtml.get("var", "")]),
         ("Mercados", [tablashtml.get("mercado", ""), bloques.get("crecimientoactivos", "")]),
         ("Datos", [tablashtml.get("calidad", ""), tablashtml.get("dcc", "")])]
motivo = "elegida automaticamente por tener el mejor Sharpe fuera de muestra" if AUTO else "fijada en los parametros"
NOTA = (f"Datos al cierre del {FECHAS[-1]:%d/%m/%Y}, reporte generado el {registro['generado']}. Precios de Yahoo Finance ajustados por dividendos y convertidos a MXN con el tipo de cambio diario; "
        f"tasa de CETES de {ORIGENC}. La estrategia recomendada es {ESTRATEGIA}, {motivo}; invierte {wr.sum():.0%} en activos con riesgo y {wobj[-1]:.0%} en CETES, con volatilidad pronosticada de "
        f"{volanual:.1%} anual, y el proximo rebalanceo mensual es el {proximo:%d/%m/%Y}. {DECISION}. "
        f"El riesgo se estima con GJR GARCH con distribucion t para cada activo y DCC para las correlaciones, reestimados cada año desde el {FECHAS[k0]:%d/%m/%Y} usando solo datos pasados. "
        f"La perdida en un mal dia es el VaR al 99% a un dia con distribucion t (uno de cada cien dias se perderia mas que eso); la de un mal mes es el VaR al 99% a 21 dias habiles con aproximacion normal. "
        f"El clima de mercado compara el riesgo pronosticado de hoy contra su historia: calma abajo del percentil 70, tension entre 70 y 90 y estres arriba de 90; hoy esta en el percentil {pct:.0%}. "
        f"El backtest rebalancea cada fin de mes desde el {FECHAS[FINMES[0]]:%d/%m/%Y} cobrando {COMISION:.2%} de comision mas IVA, y los ETFs de EUA se compran en el SIC. "
        f"Los escenarios a un año salen de 10,000 simulaciones con bootstrap estacionario. Herramienta de analisis, no asesoria financiera.")
notahtml = f"<p style='color:{T['txt2']};font-size:14px;line-height:1.6;margin:0'>{NOTA}</p>"
display(HTML(f"<div style='font-family:{FUENTE}'><b>Metodologia</b><br>{notahtml}</div>"))
cuerpo = "".join(seccion(t) + "".join(f"<div class='bloque'>{x}</div>" for x in xs if x) for t, xs in orden) + seccion("Metodologia") + notahtml
ESTILO = (".bloque{margin:14px 0;overflow-x:auto}.bloque table{min-width:680px!important}.bloque td,.bloque th{white-space:nowrap}"
          "@media (max-width:700px){.marco{padding:16px 10px 40px!important}h1{font-size:24px!important}h2{font-size:20px!important}}")
MOVIL = ('<script>window.addEventListener("load",function(){if(window.innerWidth>700)return;'
         'document.querySelectorAll(".plotly-graph-div").forEach(function(g){var an=(g.layout.annotations||[]).map(function(a){'
         'return (a.xref==="paper"&&a.x===1&&a.xanchor==="left")?Object.assign({},a,{visible:false}):a;});'
         'var tt=(g.layout.title&&g.layout.title.text)||"",p=tt.split("<br>"),sub=(p[1]||"").replace(/<[^>]+>/g,""),ln=[],cur="";'
         'sub.split(" ").forEach(function(w){if((cur+" "+w).length>44){ln.push(cur);cur=w;}else{cur=cur?cur+" "+w:w;}});if(cur)ln.push(cur);'
         'var nt=p[0]+(ln.length?"<br><span style=\'font-size:11px\'>"+ln.join("<br>")+"</span>":"");'
         'var n=g.data.filter(function(d){return d.name&&d.showlegend!==false;}).length,ex=n>1?Math.ceil(n/(n>4?1:2))*20:0;'
         'g.data.forEach(function(d,i){if(d.mode==="markers+text")Plotly.restyle(g,{mode:"markers"},[i]);});'
         'Plotly.relayout(g,{"margin.l":10,"margin.r":12,"margin.t":78+16*ln.length+ex,"margin.b":45,"legend.orientation":"h","legend.x":0,'
         '"legend.y":1.01,"legend.yanchor":"bottom","legend.font.size":10,"annotations":an.map(function(a){return a.text==="jlsd"?Object.assign({},a,{y:0,yanchor:"top",yshift:-28}):a;}),"title.text":nt,"yaxis.automargin":true,"xaxis.automargin":true,'
         '"title.font.size":15,"font.size":11});});});</script>')
reporte = f'''<!DOCTYPE html><html lang="es"><head><meta charset="utf8"><meta name="viewport" content="width=device-width, initial-scale=1">
<title>Reporte final {GENERADO:%d %m %Y}</title><style>{ESTILO}</style><script type="text/javascript">{plotly.offline.get_plotlyjs()}</script></head>
<body style="background:{T['plano']};color:{T['txt']};font-family:{FUENTE};margin:0">
<div class="marco" style="max-width:1280px;margin:0 auto;padding:28px 24px 60px">
<h1 style="font-size:32px;font-weight:700;margin:6px 0 4px">Reporte de Asignador de portafolio</h1>
{cuerpo}<div style="color:{T['mudo']};font-size:11px;text-align:right;margin-top:40px">jlsd · Jose Luis Sanchez Davila</div></div>{MOVIL}</body></html>'''
NOMBREREPORTE = f"Reporte final {GENERADO:%d %m %Y}.html"
(DIROUT / NOMBREREPORTE).write_text(reporte, encoding="utf8")
print(f"Reporte: {DIROUT / NOMBREREPORTE} · {len(FIGURAS)} graficas · {len(TABLAS)} tablas · historial con {len(hist.drop_duplicates(subset=['fechadatos', 'estrategia']))} registros")

Reporte: C:\Users\josel\OneDrive\Escritorio\AsignadorPortafolio\resultados\Reporte final 21 11 2026.html · 9 graficas · 7 tablas · historial con 1 registros
